# 08. Web Demo: Cambodian Traffic Sign Recognition (Gradio)

This notebook launches the web app in `app.py` for the **final fine-tuned ResNet-18**. **Nothing is trained here**: the app loads the real checkpoint from your Google Drive and uses the project's own model code, class list and test-time preprocessing.

**What you get:** a web page where you upload a traffic-sign image, press **Classify**, and see the predicted class, its confidence, and the top-3 predictions.

> **About the link:** with `share=True`, Gradio prints a **temporary** public link (`https://….gradio.live`). It is **not permanent**: it stops working when this Colab session ends or the runtime disconnects, and Gradio share links also expire after a limited time. Re-run the last cell to get a new link.

Run the cells from top to bottom (**Runtime → Run all**). A GPU is optional.

## 1. Google Drive and repository

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/RothRothRoth/DeepLearning_CNN.git"
PROJECT_ROOT = Path("/content/DeepLearning_CNN")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Road_Sign_Recognition_CNN")
CHECKPOINT_PATH = DRIVE_PROJECT_DIR / ("results/runs/hyperparameter_tuning/resnet18_finetuned_lr_wd_seed42/"
                                      "final_tuned_model/resnet18_finetuned_tuned_best.pt")
PROCESSED_DIR = DRIVE_PROJECT_DIR / "dataset" / "processed"

if (PROJECT_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
%cd {PROJECT_ROOT}
sys.path.insert(0, str(PROJECT_ROOT))

# ---- report missing files clearly ----
checks = {
    "app.py (in the repository)": PROJECT_ROOT / "app.py",
    "final checkpoint (on Drive)": CHECKPOINT_PATH,
    "test images folder (on Drive, used for verification and examples)": PROCESSED_DIR,
}
for label, path in checks.items():
    print(f"{'OK     ' if path.exists() else 'MISSING'}  {label}: {path}")

if not (PROJECT_ROOT / "app.py").exists():
    raise FileNotFoundError(
        "app.py is not in the cloned repository. It must be committed and pushed to GitHub first, "
        "or upload app.py into /content/DeepLearning_CNN/ using the Files panel on the left."
    )
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Final checkpoint not found:\n  {CHECKPOINT_PATH}\n"
        "Check that Drive is mounted with the account that holds the project and that "
        "notebooks/05_hyperparameter_tuning.ipynb saved final_tuned_model/resnet18_finetuned_tuned_best.pt."
    )
if not PROCESSED_DIR.is_dir():
    print("\nWARNING: test images not found, so the checkpoint cannot be verified and no example images "
          "will be shown. Uploading your own images still works.")

## 2. Install Gradio (only if missing)

In [ ]:
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr
print("Gradio", gr.__version__)

## 3. Load the final model and verify the checkpoint

`SignClassifier` (in `app.py`) rebuilds the ResNet-18 with `create_resnet18_finetuned()`, loads the checkpoint with `load_checkpoint()`, and switches to eval mode. The first time, the ImageNet weights are downloaded once; they are then replaced by the trained weights.

`verify_checkpoint` re-predicts 24 test images (the 6 misclassified ones plus 18 others) and compares them with the saved final-model predictions in the repository. The app shows the result on the page:
- ✅ **verified:** every prediction matches.
- ❌ **failed:** at least one differs, so the loaded model is not the one behind the reported results.
- ⚠️ **not verified:** the test images were not available.

In [ ]:
from app import SignClassifier, build_demo, example_images, load_reported_metrics, verify_checkpoint

classifier = SignClassifier(CHECKPOINT_PATH)
print(f"Loaded {CHECKPOINT_PATH.name} | epoch {classifier.checkpoint.get('epoch')} | device {classifier.device}")

verification = verify_checkpoint(classifier, PROCESSED_DIR)
print(f"Checkpoint verification: {verification['status'].upper()}: {verification['detail']}")
if verification["status"] == "failed":
    print("Do NOT present predictions from this model as the final model's results.")

## 4. Launch the web app

In [ ]:
demo = build_demo(classifier, verification, load_reported_metrics(), example_images(PROCESSED_DIR))
demo.launch(share=True, debug=False)
# Click the *.gradio.live link printed above. It is temporary (see the note at the top).

## How to present the web demo (about 1 minute)

**Before the presentation**
1. Run all cells once. Check that section 3 prints **VERIFIED**, and keep the tab open.
2. Open the `….gradio.live` link in a new browser tab. Optionally prepare one cropped sign image on your laptop.
3. If the session disconnected, run all cells again; the link changes every time.

**During the presentation**
1. Click an **example image**, press **Classify**, and read out the prediction, confidence and top-3.
2. Upload your own cropped sign image and classify it.
3. Point at the **Limitations** box: tight crops only, always one of 26 classes, and confidence is not a guarantee.

**Fallback:** if the link or Colab fails, use `07_demo.ipynb` or the screenshots in your slides.